In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import matplotlib.pyplot as plt

# Fetch stock data
def get_stock_data(ticker, start_date, end_date):
    stock = yf.download(ticker, start=start_date, end=end_date)

    # Compute RSI
    delta = stock['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    stock['RSI'] = 100 - (100 / (1 + rs))

    # Compute MACD
    stock['EMA_12'] = stock['Close'].ewm(span=12, adjust=False).mean()
    stock['EMA_26'] = stock['Close'].ewm(span=26, adjust=False).mean()
    stock['MACD'] = stock['EMA_12'] - stock['EMA_26']

    # Compute Bollinger Bands
    stock['Bollinger_Upper'] = stock['Close'].rolling(window=20).mean() + (2 * stock['Close'].rolling(window=20).std())
    stock['Bollinger_Lower'] = stock['Close'].rolling(window=20).mean() - (2 * stock['Close'].rolling(window=20).std())

    stock = stock[['Close', 'RSI', 'MACD', 'Bollinger_Upper', 'Bollinger_Lower']].dropna()
    return stock

# Preprocess Data
def preprocess_data(df, seq_length=30):
    scaler = MinMaxScaler(feature_range=(0.05, 0.95))
    scaled_data = scaler.fit_transform(df)

    X, y = [], []
    for i in range(len(scaled_data) - seq_length):
        X.append(scaled_data[i:i+seq_length])
        y.append(scaled_data[i+seq_length, 0])  # Predict Close price

    return np.array(X), np.array(y), scaler

# Load and preprocess dataset
df = get_stock_data("AAPL", "2022-01-01", "2025-01-01")  # Expanded data range
X, y, scaler = preprocess_data(df)

X = np.reshape(X, (X.shape[0], X.shape[1], X.shape[2]))  # Adjust for multiple indicators

# Build Enhanced LSTM Model
model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(X.shape[1], X.shape[2])),
    LSTM(128),
    Dense(1)
])
model.compile(optimizer='adam', loss=tf.keras.losses.MeanSquaredError())

# Train model
model.fit(X, y, epochs=30, batch_size=16, verbose=1)

# Save trained model
model.save("lstm_stock_model.h5")


YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed
C:\Users\navee\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 13s 88ms/step - loss: 0.0349
Epoch 2/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 5s 100ms/step - loss: 0.0021
Epoch 3/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - loss: 0.0013
Epoch 4/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step - loss: 0.0014
Epoch 5/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 4s 78ms/step - loss: 0.0012
Epoch 6/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 6s 92ms/step - loss: 0.0013
Epoch 7/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 5s 93ms/step - loss: 0.0013
Epoch 8/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 4s 74ms/step - loss: 0.0011
Epoch 9/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 6s 83ms/step - loss: 0.0011
Epoch 10/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 6s 91ms/step - loss: 0.0011
Epoch 11/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 4s 73ms/step - loss: 9.0857e-04
Epoch 12/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 5s 73ms/step - loss: 8.8804e-04
Epoch 13/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 5s 76ms/step - loss: 7.8556e-04
Epoch 14/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 5s 71ms/step - loss: 7.0102e-04
Epoch 15/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 5s 67ms/step -